# SQL inside Python with DuckDB
Python practice from [The Joinery](https://thejoinery.dev/). Run each cell with **Shift + Enter**, from the top. The explanations are on the guide page: https://thejoinery.dev/#/projects/python-duckdb

## 1. Your first query

In [ ]:
import duckdb
import pandas as pd

BASE = "https://thejoinery.dev/projects/python-data/"
headers = {"User-Agent": "Mozilla/5.0"}   # some websites block Python's default request; this sends a browser-style name
orders = pd.read_csv(BASE + "orders.csv", parse_dates=["order_date", "delivered_date"], storage_options=headers)
items = pd.read_csv(BASE + "order_items.csv", storage_options=headers)
print(duckdb.sql("SELECT COUNT(*) AS orders FROM orders").df())

## 2. GROUP BY in SQL

In [ ]:
by_province = duckdb.sql("""
    SELECT province,
           COUNT(*) AS orders,
           ROUND(SUM(order_total), 2) AS revenue
    FROM orders
    GROUP BY province
    ORDER BY revenue DESC
""").df()
print(by_province)
print(type(by_province))

## 3. Join without fan-out, using a CTE

In [ ]:
per_order = duckdb.sql("""
    WITH item_counts AS (
        SELECT order_id, COUNT(*) AS items
        FROM items
        GROUP BY order_id
    )
    SELECT o.order_id, o.order_total, i.items
    FROM orders AS o
    LEFT JOIN item_counts AS i ON o.order_id = i.order_id
    ORDER BY i.items DESC, o.order_total DESC
    LIMIT 5
""").df()
print(per_order)

## 4. The top city in each province

In [ ]:
ranked = duckdb.sql("""
    SELECT province,
           city,
           COUNT(*) AS orders,
           RANK() OVER (PARTITION BY province ORDER BY COUNT(*) DESC) AS rank_in_province
    FROM orders
    GROUP BY province, city
    QUALIFY rank_in_province = 1
    ORDER BY orders DESC
""").df()
print(ranked)

## 5. SQL first, then pandas

In [ ]:
monthly = duckdb.sql("""
    SELECT DATE_TRUNC('month', order_date) AS month,
           ROUND(SUM(order_total), 2) AS revenue
    FROM orders
    GROUP BY month
    ORDER BY month
""").df()
monthly["running_total"] = monthly["revenue"].cumsum().round(2)
print(monthly.tail(4))